# Tread-Block Cross-Section Shape Study — 3D Linear-Elastic FEM Comparison

**Objective.** Compare **8 rubber tread-block cross-section shapes**, all extruded to the same
height and all holding **exactly the same footprint area**, to rank them on:

1. **Stress build-up** — von Mises stress in the block body,
2. **Contact-face pressure distribution** — normal traction on the asphalt-facing face, and its uniformity,
3. **Deformation** — displacement under a combined normal + sliding-friction load.

Because footprint area is held fixed, every difference between the 8 results is attributable to
**shape alone** — that is the whole point of the study.

---

## What this notebook is, and is not

This is a **fast comparative screening tool**, not a validated absolute-stress prediction.
The following simplifications are deliberate, and each one is flagged again inline where it bites:

| # | Simplification | Why it is acceptable here | Where it hurts |
|---|---|---|---|
| 1 | **Linear elastic** isotropic solid instead of a hyperelastic (Mooney–Rivlin / Ogden) rubber model | Relative geometric ranking is the goal; a linear model preserves the ordering driven by geometry | Absolute stress/strain magnitudes are wrong once strains exceed ~10–20% |
| 2 | **Traction boundary condition** on the top face instead of a true unilateral contact + Coulomb friction solve | A full sliding-contact solve is nonlinear and iterative — far outside a 10-minute budget | Cannot capture contact-patch shrinkage, separation, or stick–slip |
| 3 | **Coarse tetrahedral mesh** | Keeps the whole study inside a few minutes on a Kaggle CPU | Peak stress at sharp corners is mesh-dependent (see the convergence check below) |
| 4 | **Single static load step**, no rolling, no history | Steady-state sliding snapshot | No hysteresis, no temperature, no wear evolution |

Two further points are **not** in the original specification but are established with evidence in the
*Verification* section below, because ignoring them would make the comparison unreliable:

- **Volumetric locking.** At $\nu = 0.49$ the rubber is nearly incompressible, and *linear* (Tet4/P1)
  tetrahedra lock badly — they come out several times too stiff and **do not converge** under mesh
  refinement. The mesh stays Tet4 as specified, but the *element* is raised to **quadratic (P2)**,
  which removes the locking at negligible cost on meshes this small. `ELEMENT_ORDER` is a single
  switch if you want to reproduce the P1 behaviour.
- **The specified load is far outside the linear-elastic range.** 500 N on a 100 mm² face is a
  nominal compressive stress of 5 MPa against a 5 MPa modulus, i.e. **~100% nominal strain**
  (and ~270% nominal shear strain). This is quantified in the *Load sanity check* cell.
  Because the model is *linear*, every result scales **exactly linearly** with the load, so the
  **relative ranking between shapes — the actual deliverable — is completely load-independent.**
  Absolute displacement and stress numbers, however, should be read as *"per unit of this load case"*
  and not as physical predictions.

## 1. Install and imports

`shapely` is required for the polygon-area verification, so it is added to the install list.
`pyvista` is **not** installed: every figure here is static `matplotlib`, which renders reliably
on Kaggle without a virtual framebuffer and keeps the install fast.

**Runtime.** End-to-end this takes roughly **5 minutes of compute** plus 1–2 minutes of installs on a
Kaggle CPU instance — comfortably inside a 10–15 minute budget. Roughly half of it is the two
verification studies in section 6, which are the parts that make the comparison defensible. If you
need it faster, raise `MESH_SIZE` to 1.5 mm; if you want tighter convergence, lower it to 1.0 mm and
expect roughly 2.5× the solve time.

In [ ]:
!pip install -q gmsh meshio scikit-fem shapely

In [ ]:
import math, os, time, tempfile, textwrap
import numpy as np
import pandas as pd
import gmsh, meshio
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from shapely.geometry import Polygon

import skfem
from skfem import (MeshTet, Basis, FacetBasis, ElementVector,
                   ElementTetP1, ElementTetP2, BilinearForm, LinearForm,
                   asm, condense, solve)
from skfem.helpers import sym_grad, ddot, trace

plt.rcParams.update({"figure.dpi": 96, "font.size": 9, "axes.titlesize": 10})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

gmsh.initialize(); _v = gmsh.option.getString("General.Version"); gmsh.finalize()
print(f"numpy {np.__version__} | scipy-backed scikit-fem {skfem.__version__} | "
      f"gmsh {_v} | meshio {meshio.__version__} | pandas {pd.__version__}")

## 2. Parameters

Everything tunable lives here. Units are **N, mm, MPa** throughout
(1 N/mm² = 1 MPa), so a "pressure" and a "stress" are the same number.

In [ ]:
# ---- material (linear-elastic approximation of a tread compound) -------------
E_MOD   = 5.0      # MPa   Young's modulus (apparent stiffness of tread rubber)
NU      = 0.49     # -     Poisson's ratio, near-incompressible

# ---- geometry ---------------------------------------------------------------
BLOCK_H = 10.0     # mm    extrusion height of every block
AREA_T  = 100.0    # mm^2  target footprint area, IDENTICAL for all 8 shapes

# ---- loading ----------------------------------------------------------------
LOAD_N  = 500.0    # N     total vertical load on the block
MU_FRIC = 0.9      # -     Coulomb friction coefficient (rubber on asphalt)
SLIDE_DIR = 0      # 0=x, 1=y : in-plane direction of the friction drag

# ---- discretisation ---------------------------------------------------------
MESH_SIZE     = 1.2   # mm  target tet edge length (coarse, per the speed budget)
ELEMENT_ORDER = 2     # 1 = Tet4/P1, 2 = Tet10/P2 on the same Tet4 mesh.
                      # P2 is the default: P1 locks at nu=0.49 (see Verification).

# ---- derived Lame parameters ------------------------------------------------
LAMBDA = E_MOD * NU / ((1.0 + NU) * (1.0 - 2.0 * NU))
G_MOD  = E_MOD / (2.0 * (1.0 + NU))
K_BULK = E_MOD / (3.0 * (1.0 - 2.0 * NU))

print(f"lambda = {LAMBDA:10.4f} MPa")
print(f"G      = {G_MOD:10.4f} MPa")
print(f"K      = {K_BULK:10.4f} MPa   (K/G = {K_BULK/G_MOD:.1f}  -> near-incompressible)")

### 2b. Load sanity check

A quick dimensional check before solving anything, because it governs how the numbers
downstream must be read.

In [ ]:
p_nom = LOAD_N / AREA_T           # MPa, nominal normal pressure
q_nom = MU_FRIC * p_nom           # MPa, nominal friction shear traction

eps_ax  = p_nom / E_MOD           # nominal axial strain
gam_sh  = q_nom / G_MOD           # nominal shear strain

print(f"nominal normal pressure  p = {p_nom:6.3f} MPa   (E = {E_MOD} MPa)")
print(f"nominal friction shear   q = {q_nom:6.3f} MPa   (G = {G_MOD:.3f} MPa)")
print(f"  -> nominal axial strain  p/E = {eps_ax*100:7.1f} %")
print(f"  -> nominal shear strain  q/G = {gam_sh*100:7.1f} %")

if max(eps_ax, gam_sh) > 0.20:
    print(textwrap.dedent(f"""
    !! LINEARITY WARNING ----------------------------------------------------
       Nominal strains of {eps_ax*100:.0f}% / {gam_sh*100:.0f}% are far beyond the ~10-20% range
       where linear elasticity is a fair approximation of rubber. Absolute
       displacements and stresses printed below are therefore NOT physical
       predictions.

       This does NOT invalidate the study. The model is strictly linear, so
       every field scales exactly linearly with LOAD_N: doubling the load
       doubles every displacement and every stress. The RANKING of the 8
       shapes -- which is the deliverable -- is therefore identical at any
       load level, including a physically sensible one.

       To get physically meaningful magnitudes, lower LOAD_N (e.g. 50 N gives
       ~10% nominal strain) or move to a hyperelastic solver. Rankings will
       not change.
    ------------------------------------------------------------------------"""))

## 3. The 8 cross-sections, area-matched

Each shape is generated at **unit scale**, its area measured, and then every in-plane
dimension is multiplied by $s=\sqrt{A_\text{target}/A_\text{unit}}$.

Area scales exactly as $s^2$ under a uniform in-plane scaling — **including the filleted
shapes**, provided the fillet radius is itself defined as a *fraction of a feature size* and so
scales with the shape. That makes the area match exact in one step rather than iterative.

For the two filleted shapes and the ellipse the area is measured by **OpenCASCADE**
(`gmsh.model.occ.getMass`) rather than by `shapely`, because a filleted outline is bounded by true
circular arcs that a polygon approximation would misrepresent.

In [ ]:
# --------------------------------------------------------------------------
# Each generator returns an ordered list of 2D boundary points for a given
# in-plane scale s. Fillets are applied later, in OCC, as a fraction of a
# named feature size so that they scale with s.
# --------------------------------------------------------------------------

def poly_square(s=1.0):
    a = s
    return [(-a/2, -a/2), (a/2, -a/2), (a/2, a/2), (-a/2, a/2)]

def poly_regular(n, s=1.0):
    """Regular n-gon of circumradius s, flat-ish side down."""
    return [(s*math.cos(2*math.pi*k/n + math.pi/2),
             s*math.sin(2*math.pi*k/n + math.pi/2)) for k in range(n)]

def poly_trapezoid(s=1.0):
    """Isosceles trapezoid: top width 0.6 x bottom, height 0.9 x bottom."""
    b, t, h = s, 0.6*s, 0.9*s
    return [(-b/2, -h/2), (b/2, -h/2), (t/2, h/2), (-t/2, h/2)]

def poly_rect(s=1.0):
    """Rectangle, aspect 1.5:1. Basis for the rounded rectangle."""
    W, H = 1.5*s, 1.0*s
    return [(-W/2, -H/2), (W/2, -H/2), (W/2, H/2), (-W/2, H/2)]

def poly_chevron(s=1.0):
    """
    V / arrow tread lug. Half-span w=s, apex rise 1.2w, band thickness t=0.5w.
    The inner apex (0, 1.2w - t) is a RE-ENTRANT corner -> genuine stress
    singularity, which is exactly the feature this study is meant to expose.
    """
    w = s; h = 1.2*w; t = 0.5*w
    return [(-w, 0.0), (0.0, h), (w, 0.0), (w, -t), (0.0, h - t), (-w, -t)]

# feature size used to set each fillet radius (at unit scale)
RRECT_FILLET_FRAC = 0.15   # of the SHORT side  (spec: ~15%)
RCHEV_FILLET_FRAC = 0.12   # of the band thickness (spec: ~10-15%)

In [ ]:
# --------------------------------------------------------------------------
# OCC geometry construction. Fillets are applied to the extruded solid's
# VERTICAL edges: that rounds the cross-section identically at every height,
# which is what "rounded cross-section" means for a prismatic block, and it
# uses gmsh's well-supported 3D fillet rather than a 2D wire fillet.
# --------------------------------------------------------------------------

def _vertical_edges(vol_tag, height, tol=1e-6):
    """Curve tags of the prism's side edges (those spanning the full height)."""
    out = []
    for dim, tag in gmsh.model.getBoundary([(3, vol_tag)], oriented=False):
        for _, ctag in gmsh.model.getBoundary([(dim, tag)], oriented=False):
            ctag = abs(ctag)
            if ctag in out:
                continue
            x0, y0, z0, x1, y1, z1 = gmsh.model.getBoundingBox(1, ctag)
            if abs((z1 - z0) - height) < tol and (x1 - x0) < tol and (y1 - y0) < tol:
                out.append(ctag)
    return sorted(set(out))


def build_solid(pts=None, ellipse=None, fillet_r=0.0, height=BLOCK_H):
    """Build one extruded (optionally filleted) solid in the CURRENT gmsh model."""
    if ellipse is not None:
        a, b = ellipse                      # addEllipse needs r1 >= r2
        c = gmsh.model.occ.addEllipse(0, 0, 0, max(a, b), min(a, b))
        surf = gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop([c])])
    else:
        pt = [gmsh.model.occ.addPoint(x, y, 0.0) for x, y in pts]
        ln = [gmsh.model.occ.addLine(pt[i], pt[(i + 1) % len(pt)]) for i in range(len(pt))]
        surf = gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop(ln)])

    ext = gmsh.model.occ.extrude([(2, surf)], 0, 0, height)
    vol = [t for d, t in ext if d == 3][0]
    gmsh.model.occ.synchronize()

    if fillet_r > 0.0:
        res = gmsh.model.occ.fillet([vol], _vertical_edges(vol, height),
                                    [fillet_r], removeVolume=True)
        gmsh.model.occ.synchronize()
        vol = [t for d, t in res if d == 3][0]
    return vol


def occ_geom(pts=None, ellipse=None, fillet_r=0.0):
    """
    Exact cross-section area AND perimeter from OpenCASCADE.

        area      = volume / height
        perimeter = lateral surface area / height
                  = (total surface area - top - bottom) / height

    Both are exact for filleted outlines and for the ellipse, where a polygon
    approximation would be wrong. The perimeter feeds the shape factor
    Q = P^2 / (4*pi*A), which turns out to explain most of the ranking.
    """
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("probe")
    try:
        vol = build_solid(pts=pts, ellipse=ellipse, fillet_r=fillet_r)
        area = gmsh.model.occ.getMass(3, vol) / BLOCK_H
        total_surf = sum(gmsh.model.occ.getMass(2, abs(t))
                         for d, t in gmsh.model.getBoundary([(3, vol)], oriented=False))
        return area, (total_surf - 2.0 * area) / BLOCK_H
    finally:
        gmsh.finalize()


def occ_area(pts=None, ellipse=None, fillet_r=0.0):
    return occ_geom(pts=pts, ellipse=ellipse, fillet_r=fillet_r)[0]

In [ ]:
# --------------------------------------------------------------------------
# Build the 8 area-matched shape specifications.
# A spec is a dict the mesher understands: {pts | ellipse, fillet_r}.
# --------------------------------------------------------------------------

def _scaled_polygon_spec(gen, target=AREA_T):
    """Sharp polygon: shapely gives the area analytically, one-shot exact scale."""
    s = math.sqrt(target / Polygon(gen(1.0)).area)
    return {"pts": gen(s), "fillet_r": 0.0, "scale": s}


def _scaled_filleted_spec(gen, fillet_frac, feature_of, target=AREA_T):
    """
    Filleted polygon: measure the filleted area at unit scale in OCC, then apply
    the sqrt scale to EVERY in-plane length including the fillet radius. Because
    the whole outline (radius included) scales by s, the area scales exactly by
    s^2 -- so this single step is exact, not iterative.
    """
    r1 = fillet_frac * feature_of(1.0)
    a1 = occ_area(pts=gen(1.0), fillet_r=r1)
    s = math.sqrt(target / a1)
    return {"pts": gen(s), "fillet_r": fillet_frac * feature_of(s), "scale": s}


# ellipse, aspect a/b = 1.5 -> pi*a*b = A  (analytic, no fillet needed)
_b = math.sqrt(AREA_T / (math.pi * 1.5)); _a = 1.5 * _b

SHAPES = {
    "Square":            _scaled_polygon_spec(poly_square),
    "Hexagon":           _scaled_polygon_spec(lambda s: poly_regular(6, s)),
    "Pentagon":          _scaled_polygon_spec(lambda s: poly_regular(5, s)),
    "Trapezoid":         _scaled_polygon_spec(poly_trapezoid),
    "Rounded rect":      _scaled_filleted_spec(poly_rect, RRECT_FILLET_FRAC,
                                               lambda s: 1.0 * s),   # short side
    "Ellipse":           {"ellipse": (_a, _b), "fillet_r": 0.0, "scale": 1.0},
    "Chevron":           _scaled_polygon_spec(poly_chevron),
    "Rounded chevron":   _scaled_filleted_spec(poly_chevron, RCHEV_FILLET_FRAC,
                                               lambda s: 0.5 * s),   # band thickness
}

SHAPE_ORDER = list(SHAPES)
print(f"{len(SHAPES)} shapes defined:", ", ".join(SHAPE_ORDER))

### 3b. Area-matching verification

Every shape must land within 1% of the 100 mm² target. The `OCC area` column is the
authoritative one — it is the actual area of the solid that will be meshed.

In [ ]:
rows = []
for name in SHAPE_ORDER:
    sp = SHAPES[name]
    a_occ, perim = occ_geom(pts=sp.get("pts"), ellipse=sp.get("ellipse"),
                            fillet_r=sp["fillet_r"])
    a_shp = Polygon(sp["pts"]).area if "pts" in sp else np.nan   # sharp outline only
    rows.append({
        "shape": name,
        "shapely area (sharp outline)": a_shp,
        "OCC area [mm^2]": a_occ,
        "error vs target [%]": 100.0 * (a_occ - AREA_T) / AREA_T,
        "perimeter [mm]": perim,
        "shape factor Q [-]": perim ** 2 / (4.0 * math.pi * a_occ),
        "fillet r [mm]": sp["fillet_r"],
        "in-plane scale": sp["scale"],
    })

area_df = pd.DataFrame(rows).set_index("shape")
SHAPE_Q = area_df["shape factor Q [-]"]
SHAPE_P = area_df["perimeter [mm]"]

display(area_df.style.format({
    "shapely area (sharp outline)": "{:.4f}", "OCC area [mm^2]": "{:.4f}",
    "error vs target [%]": "{:+.3e}", "perimeter [mm]": "{:.3f}",
    "shape factor Q [-]": "{:.4f}", "fillet r [mm]": "{:.4f}",
    "in-plane scale": "{:.4f}"})
    .background_gradient(cmap="Oranges", subset=["shape factor Q [-]"])
    .set_caption("Area matching (target = 100.0000 mm^2) + shape compactness"))

worst = area_df["error vs target [%]"].abs().max()
assert worst < 1.0, f"area matching failed: worst error {worst:.3f}%"
print(f"OK - all 8 areas within {worst:.2e} % of target (tolerance 1%).")
print("\nNotes:")
print(" * 'shapely area' is the SHARP polygon; for the two filleted shapes it is")
print("   intentionally larger than the OCC area, since fillets remove corner material.")
print(" * Q = P^2/(4*pi*A) is the isoperimetric shape factor: Q = 1 for a circle and")
print("   grows as an outline becomes less compact (more perimeter for the same area).")
print("   Area is held fixed across all 8 shapes, so Q is the single scalar that best")
print("   summarises 'how spread out' each cross-section is. Keep an eye on it -- it")
print("   turns out to predict the stress ranking better than corner sharpness does.")

In [ ]:
# --- visual check: the 8 outlines, drawn to the same scale --------------------
fig, axes = plt.subplots(2, 4, figsize=(14, 6.6))
for ax, name in zip(axes.ravel(), SHAPE_ORDER):
    sp = SHAPES[name]
    if "ellipse" in sp:
        th = np.linspace(0, 2*np.pi, 200)
        xs, ys = sp["ellipse"][0]*np.cos(th), sp["ellipse"][1]*np.sin(th)
    else:
        p = np.array(sp["pts"] + [sp["pts"][0]])
        xs, ys = p[:, 0], p[:, 1]
    ax.fill(xs, ys, alpha=0.30, color="tab:blue")
    ax.plot(xs, ys, color="tab:blue", lw=1.6)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlim(-11, 11); ax.set_ylim(-11, 13)
    r = sp["fillet_r"]
    ax.set_title(f"{name}\nA = {AREA_T:.1f} mm^2" + (f", r = {r:.2f} mm" if r else ""))
fig.suptitle("The 8 cross-sections — identical footprint area, sharp outlines shown "
             "(fillets applied in OCC)", y=1.00)
fig.tight_layout(); plt.show()

## 4. Meshing

One generic function: build the OCC solid → mesh with linear tetrahedra →
hand the result to `scikit-fem` through `meshio`.

The top and bottom faces are identified **geometrically** by their $z$ coordinate rather than by
gmsh physical groups. For a prism extruded along $z$ this is exact and robust, and it survives the
fillet operation (which only touches the vertical edges).

> **Equal mesh density is a requirement, not a detail.** Peak stress at a corner is a function of how
> finely that corner is meshed. If one shape is meshed more finely than another, it will report
> higher peak stress *for that reason alone*, and the comparison becomes meaningless. Curvature-adaptive
> sizing is therefore switched off — it silently refines the filleted shapes 3–4× harder than the sharp
> ones and inverts the result. All 8 shapes get one uniform target size, and because they all have the
> same footprint area, that yields near-equal element density. This is checked explicitly after the run.

In [ ]:
def make_mesh(spec, h_size=MESH_SIZE):
    """OCC solid -> Tet4 mesh -> skfem MeshTet. Returns (mesh, n_tets)."""
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("blk")
    try:
        build_solid(pts=spec.get("pts"), ellipse=spec.get("ellipse"),
                    fillet_r=spec["fillet_r"])

        gmsh.option.setNumber("Mesh.MeshSizeMax", h_size)
        gmsh.option.setNumber("Mesh.MeshSizeMin", h_size * 0.5)
        # ---------------------------------------------------------------
        # Curvature-driven sizing is deliberately DISABLED.
        #
        # It is tempting to switch it on so that fillets get extra elements.
        # Doing so wrecks the study: a small fillet radius (0.6 mm here)
        # drags the local size down to the floor and the rounded shapes end
        # up with 3-4x the element density of the sharp ones. Peak stress
        # grows with mesh density at any corner, so the rounded shapes then
        # report HIGHER peak stress than the sharp ones -- the exact opposite
        # of the truth, produced entirely by the mesh rather than the physics.
        #
        # Because footprint area is identical for all 8 shapes, a uniform
        # target size gives near-identical element density everywhere, which
        # is what makes cross-shape comparison legitimate. The density spread
        # is asserted below.
        # ---------------------------------------------------------------
        gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
        gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 1)
        gmsh.option.setNumber("Mesh.Algorithm3D", 1)          # Delaunay
        gmsh.option.setNumber("Mesh.Optimize", 1)
        gmsh.model.mesh.generate(3)

        path = os.path.join(tempfile.gettempdir(), "block.msh")
        gmsh.write(path)
    finally:
        gmsh.finalize()

    mio = meshio.read(path)
    tets = np.vstack([c.data for c in mio.cells if c.type == "tetra"])
    mesh = MeshTet(np.ascontiguousarray(mio.points.T),
                   np.ascontiguousarray(tets.T))
    return mesh, tets.shape[0]


def face_facets(mesh, tol=1e-6):
    """Facet indices of the bottom (z=0) and top (z=BLOCK_H) faces."""
    zmax = mesh.p[2].max()
    bot = mesh.facets_satisfying(lambda x: x[2] < tol)
    top = mesh.facets_satisfying(lambda x: x[2] > zmax - tol)
    return bot, top

## 5. Boundary conditions and the two load cases

**Bottom face ($z=0$, bonded to the undertread):** fully fixed, $\mathbf{u}=\mathbf{0}$ — Dirichlet.

**Side faces:** traction-free — the natural BC, so nothing to impose.

**Top face ($z=10$ mm, against the asphalt):** this is where the contact simplification lives.
Two load cases are run, and the difference between them matters:

### Load Case A — uniform traction (as specified)

Both tractions are applied as a **Neumann (surface traction) term in the weak form**, not as point loads:

$$\int_{\Gamma_\text{top}} \mathbf{t}\cdot\mathbf{v}\,\mathrm{d}s,
\qquad \mathbf{t} = \big(\mu p,\; 0,\; -p\big),\qquad p = \frac{F}{A_\text{face}}$$

### Load Case B — rigid platen (supplementary)

Load Case A applies a *uniform* pressure. It therefore **largely prescribes the answer** to one of
the three questions this study asks: if you impose a uniform normal traction, the recovered normal
traction on that face is uniform by construction, up to corner singularities and discretisation
error. It ranks stress and deformation well, but it is a weak instrument for comparing **contact
pressure distribution**.

Load Case B fixes that by modelling the asphalt as a **rigid flat platen**: the top face is
constrained to stay flat and move down as a unit ($u_z = -\delta$, with $u_x,u_y$ free to slide),
while the same friction shear traction is applied. Now the pressure distribution is an **output**,
and it reproduces the classic flat-punch result — pressure rising sharply toward the edges and
corners of the footprint. This is the case that actually discriminates shapes on pressure uniformity.

$\delta$ is not guessed. The problem is linear, so the solution is decomposed exactly:

$$\mathbf{u} = \delta\,\mathbf{u}_A + \mathbf{u}_B$$

where $\mathbf{u}_A$ is the unit-indentation solution (no shear) and $\mathbf{u}_B$ is the
shear-only solution with $u_z=0$ on top. With $R(\delta)=\delta R_A + R_B$ the total vertical
reaction, $\delta$ follows in closed form from $R = F$. Both share one Dirichlet set and one
stiffness matrix, so it costs **two right-hand sides, not two factorisations**.

**Load Case B caveat:** a bonded platen can transmit *tension*. The friction drag applies a moment
about the bonded base, so the trailing edge of the footprint is pulled into tension where a real
tyre block would simply lift off. Negative pressure is reported as `p_min` and read as
*"this region would have separated"*, not as a physical suction.

In [ ]:
# --------------------------------------------------------------------------
# Weak forms. Isotropic linear elasticity:
#     sigma = 2*G*eps + lambda*tr(eps)*I
# --------------------------------------------------------------------------

@BilinearForm
def stiffness_form(u, v, w):
    eu, ev = sym_grad(u), sym_grad(v)
    return 2.0 * G_MOD * ddot(eu, ev) + LAMBDA * trace(eu) * trace(ev)


def sigma_of(eps):
    """Cauchy stress from a skfem strain array of shape (3, 3, nelem, nqp)."""
    tr = eps[0, 0] + eps[1, 1] + eps[2, 2]
    s = 2.0 * G_MOD * np.asarray(eps)
    for i in range(3):
        s[i, i] = s[i, i] + LAMBDA * tr
    return s


def von_mises_of(s):
    """von Mises invariant of a (3, 3, ...) stress array."""
    p = (s[0, 0] + s[1, 1] + s[2, 2]) / 3.0
    d = np.array(s, copy=True)
    for i in range(3):
        d[i, i] = d[i, i] - p
    return np.sqrt(1.5 * np.einsum("ij...,ij...->...", d, d))


def weighted_percentile(values, weights, q):
    """
    Percentile weighted by element volume / facet area.

    Plain np.percentile treats a sliver tet at a singular corner the same as a
    bulk element, which biases the statistic toward wherever the mesh happens to
    be dense. Weighting by measure makes the percentile a property of the SOLID,
    not of the mesh -- essential when comparing 8 different meshes to each other.
    """
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    idx = np.argsort(values)
    v, wt = values[idx], weights[idx]
    c = np.cumsum(wt) - 0.5 * wt
    return float(np.interp(q / 100.0 * wt.sum(), c, v))

In [ ]:
def build_system(mesh, order=ELEMENT_ORDER):
    """Assemble K, the top-face basis, and the fixed-base DOFs — shared by both cases."""
    elem = ElementVector(ElementTetP2() if order == 2 else ElementTetP1())
    basis = Basis(mesh, elem, intorder=2)
    bot, top = face_facets(mesh)
    fb_top = FacetBasis(mesh, elem, facets=top)

    K = asm(stiffness_form, basis)
    dofs_bot = basis.get_dofs(facets=bot).all()
    area_top = float(np.sum(fb_top.dx))       # integrate 1 over the top facets
    return dict(basis=basis, fb_top=fb_top, K=K, dofs_bot=dofs_bot,
                top=top, bot=bot, area_top=area_top)


def solve_case_A(sys):
    """Uniform normal + friction traction on the top face (the specified case)."""
    p = LOAD_N / sys["area_top"]
    q = MU_FRIC * p

    @LinearForm
    def traction(v, w):
        # -p on z (compressive) and +q along the sliding direction
        return -p * v[2] + q * v[SLIDE_DIR]

    f = asm(traction, sys["fb_top"])
    x = solve(*condense(sys["K"], f, D=sys["dofs_bot"]))
    return x, dict(p_applied=p, q_applied=q)


def solve_case_B(sys):
    """Rigid flat platen: u_z prescribed on top, in-plane free, same shear drag."""
    basis, K = sys["basis"], sys["K"]
    p = LOAD_N / sys["area_top"]
    q = MU_FRIC * p

    dofs_top_z = basis.get_dofs(facets=sys["top"]).all("u^3")
    D = np.unique(np.concatenate([sys["dofs_bot"], dofs_top_z]))

    @LinearForm
    def shear(v, w):
        return q * v[SLIDE_DIR]

    f_shear = asm(shear, sys["fb_top"])

    # u_A : unit downward indentation, no shear
    xa = np.zeros(basis.N); xa[dofs_top_z] = -1.0; xa[sys["dofs_bot"]] = 0.0
    u_A = solve(*condense(K, np.zeros(basis.N), x=xa, D=D))
    # u_B : shear only, top held at u_z = 0
    u_B = solve(*condense(K, f_shear, D=D))

    # vertical reaction at the platen = -sum of residual at the constrained z-DOFs
    R_A = -float(np.sum((K @ u_A)[dofs_top_z]))
    R_B = -float(np.sum((K @ u_B - f_shear)[dofs_top_z]))
    delta = (LOAD_N - R_B) / R_A                      # exact, by linearity

    x = delta * u_A + u_B
    R = -float(np.sum((K @ x - f_shear)[dofs_top_z]))
    return x, dict(p_applied=p, q_applied=q, indentation=delta, reaction_N=R)

In [ ]:
def post_process(sys, x):
    """Displacement, von Mises and top-face pressure fields + scalar metrics."""
    basis, fb = sys["basis"], sys["fb_top"]

    # ---- displacement (at mesh vertices, so P1 and P2 are comparable) --------
    u_nod = x[basis.nodal_dofs]                       # (3, nverts)
    u_mag = np.linalg.norm(u_nod, axis=0)

    # ---- von Mises, element-wise, volume-weighted statistics ----------------
    eps = sym_grad(basis.interpolate(x))
    vm_qp = von_mises_of(sigma_of(eps))               # (nelem, nqp)
    vm_el = vm_qp.mean(axis=1)
    vol_el = np.sum(basis.dx, axis=1)                 # element volumes

    # ---- top-face normal pressure -------------------------------------------
    # Recover sigma at the facet quadrature points and project onto the face
    # normal: p = -n . sigma . n  (positive = compression).
    sig_f = sigma_of(sym_grad(fb.interpolate(x)))
    n = fb.normals
    p_qp = -np.einsum("i...,ij...,j...->...", n, sig_f, n)
    p_fac = p_qp.mean(axis=1)                         # per top facet
    a_fac = np.sum(fb.dx, axis=1)                     # facet areas

    p_mean = float(np.sum(p_fac * a_fac) / np.sum(a_fac))
    p_std = float(np.sqrt(np.sum((p_fac - p_mean) ** 2 * a_fac) / np.sum(a_fac)))

    metrics = {
        "u_max [mm]":       float(u_mag.max()),
        "u_mean [mm]":      float(u_mag.mean()),
        "vM_max [MPa]":     float(vm_el.max()),
        "vM_p95 [MPa]":     weighted_percentile(vm_el, vol_el, 95),
        "p_max [MPa]":      float(p_fac.max()),
        "p_p95 [MPa]":      weighted_percentile(p_fac, a_fac, 95),
        "p_mean [MPa]":     p_mean,
        "p_min [MPa]":      float(p_fac.min()),
        "p_CoV [-]":        p_std / p_mean if p_mean != 0 else np.nan,
        "Fz check [N]":     float(np.sum(p_fac * a_fac)),
    }
    fields = dict(u_nod=u_nod, u_mag=u_mag, vm_el=vm_el, p_fac=p_fac, a_fac=a_fac)
    return metrics, fields

In [ ]:
def run_shape(name, spec, h_size=MESH_SIZE, order=ELEMENT_ORDER, verbose=True,
              mesh=None, n_tet=None):
    """Full pipeline for one shape: mesh -> assemble -> solve A and B -> metrics."""
    t0 = time.time()
    if mesh is None:
        mesh, n_tet = make_mesh(spec, h_size=h_size)
    t_mesh = time.time() - t0

    t0 = time.time()
    sys = build_system(mesh, order=order)
    xA, infoA = solve_case_A(sys)
    xB, infoB = solve_case_B(sys)
    t_solve = time.time() - t0

    mA, fA = post_process(sys, xA)
    mB, fB = post_process(sys, xB)

    common = {"shape": name, "area_top [mm^2]": sys["area_top"],
              "n_nodes": mesh.p.shape[1], "n_tets": n_tet, "n_dof": sys["basis"].N,
              "t_mesh [s]": t_mesh, "t_solve [s]": t_solve}
    if verbose:
        print(f"  {name:16s} tets={n_tet:6d} dof={sys['basis'].N:7d} "
              f"A_top={sys['area_top']:8.4f} mesh={t_mesh:5.2f}s solve={t_solve:5.2f}s "
              f"| A: u_max={mA['u_max [mm]']:8.2f} vM_p95={mA['vM_p95 [MPa]']:7.2f} "
              f"| B: delta={infoB['indentation']:6.3f} R={infoB['reaction_N']:7.1f}N")
    return dict(mesh=mesh, sys=sys, common=common,
                A=dict(x=xA, metrics=mA, fields=fA, info=infoA),
                B=dict(x=xB, metrics=mB, fields=fB, info=infoB))

## 6. Verification

Two checks, run once on the chevron (the most demanding shape — it has a re-entrant corner):

1. **Element order / volumetric locking** — does P1 give trustworthy answers at $\nu=0.49$?
2. **Mesh convergence** — which reported metrics are converged, and which are not?

This costs well under a minute and determines how the results table should be read.

In [ ]:
probe = SHAPES["Chevron"]
conv_rows = []
for h in [2.5, 2.0, 1.5, 1.2]:
    for order in (1, 2):
        r = run_shape("Chevron", probe, h_size=h, order=order, verbose=False)
        conv_rows.append({
            "h [mm]": h, "order": f"P{order}", "n_tets": r["common"]["n_tets"],
            "n_dof": r["common"]["n_dof"],
            "u_max [mm]": r["A"]["metrics"]["u_max [mm]"],
            "vM_p95 [MPa]": r["A"]["metrics"]["vM_p95 [MPa]"],
            "vM_max [MPa]": r["A"]["metrics"]["vM_max [MPa]"],
            "p_max [MPa]": r["A"]["metrics"]["p_max [MPa]"],
            "t_solve [s]": r["common"]["t_solve [s]"],
        })

conv = pd.DataFrame(conv_rows)
display(conv.pivot(index="h [mm]", columns="order",
                   values=["n_dof", "u_max [mm]", "vM_p95 [MPa]", "p_max [MPa]", "t_solve [s]"])
        .style.format("{:.3f}").set_caption(
            "Convergence: P1 vs P2 under mesh refinement (Load Case A, chevron)"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
for ax, col, ttl in zip(
        axes, ["u_max [mm]", "vM_p95 [MPa]", "p_max [MPa]"],
        ["max displacement (converging?)", "95th-pct von Mises (converging?)",
         "max contact pressure (singular)"]):
    for order, mk in [("P1", "o--"), ("P2", "s-")]:
        d = conv[conv["order"] == order].sort_values("h [mm]", ascending=False)
        ax.plot(d["h [mm]"], d[col], mk, label=order)
    ax.invert_xaxis(); ax.set_xlabel("mesh size h [mm]  (refining ->)")
    ax.set_ylabel(col); ax.set_title(ttl); ax.grid(alpha=0.3); ax.legend()
fig.suptitle("Verification — element order and mesh convergence (chevron)", y=1.03)
fig.tight_layout(); plt.show()

p1 = conv[conv["order"] == "P1"].sort_values("h [mm]", ascending=False)["u_max [mm]"].to_numpy()
p2 = conv[conv["order"] == "P2"].sort_values("h [mm]", ascending=False)["u_max [mm]"].to_numpy()
drift = lambda a: 100.0 * abs(a[-1] - a[-2]) / abs(a[-1])
print(f"u_max drift over the last refinement step:  P1 {drift(p1):5.1f} %   P2 {drift(p2):5.1f} %")
print(f"u_max at the finest mesh:                   P1 {p1[-1]:8.2f} mm  P2 {p2[-1]:8.2f} mm  "
      f"(P1 is {p2[-1]/p1[-1]:.1f}x too stiff)")

**Reading the verification.**

- **P1 locks, and it does not converge.** Its `u_max` is still climbing steadily at the finest mesh
  while P2 has already flattened out, and P1 remains several times stiffer than the converged
  answer. This is textbook **volumetric locking** of linear tetrahedra at $\nu\to 0.5$: the constant
  strain field inside a Tet4 cannot represent an isochoric deformation, so the element resists any
  motion at all. Tet4 meshes with P1 elements are simply not usable for near-incompressible rubber.
  **Hence `ELEMENT_ORDER = 2`** — the same Tet4 mesh from gmsh, with a quadratic basis on it.
- **`u_max` and `vM_p95` are converged** to a few percent with P2 at `MESH_SIZE = 1.2 mm`. These
  are the metrics to trust and to rank shapes by.
- **`vM_max` and `p_max` are *not* converged and never will be.** The chevron's re-entrant corner is
  a genuine stress singularity: the exact elasticity solution has *infinite* stress there, so every
  refinement returns a larger peak. Reported maxima are therefore **mesh-resolution artefacts** and
  must be compared only between meshes of comparable density — which is why every production run
  below uses one common `MESH_SIZE`. The volume-weighted **95th percentile** is the robust
  stand-in, and is why the specification asked for it.

### 6b. Sharp vs filleted corner — a convergence argument, not a number

"Rounding the corner lowers peak stress" cannot be established by quoting two peak-stress numbers
from one mesh, because the peak at a *sharp* re-entrant corner is not a number at all — it is a
divergent quantity that grows every time you refine.

The correct statement is about **convergence behaviour**, and it is directly testable: refine both
chevron variants at the corner and watch what the peak does.

- **Sharp chevron** — $\sigma_{max}$ should keep climbing without limit. The elasticity solution is
  genuinely singular at a re-entrant corner ($\sigma \sim r^{\lambda-1}$ with $\lambda<1$).
- **Rounded chevron** — $\sigma_{max}$ should approach a finite plateau once the mesh resolves the fillet.

Two things have to be right for this test to mean anything:

1. **Refine locally, not globally.** The singularity is local; a global refinement fine enough to
   resolve it would be unaffordable. The size field is fine at the apex and coarsens away from it.
   The refinement cone must be wide enough to cover the *fillet*, which sits one radius away from the
   nominal apex — too tight a cone refines the sharp corner while starving the rounded one, and
   manufactures the wrong answer.
2. **Sample only near the corner.** The bonded base edge is *also* a singularity (bonded-to-free
   transition). A global maximum is usually located there, and it would drown out the corner effect
   entirely. The peak is therefore taken inside a small ball around the apex at mid-height.

In [ ]:
def reentrant_vertex(pts):
    """Locate the re-entrant (reflex, interior angle > 180 deg) vertex of a polygon."""
    P = np.asarray(pts, float)
    orient = np.sign(sum((P[(i + 1) % len(P), 0] - P[i, 0]) *
                         (P[(i + 1) % len(P), 1] + P[i, 1]) for i in range(len(P))))
    for i in range(len(P)):
        a, b, c = P[i - 1], P[i], P[(i + 1) % len(P)]
        # 2D scalar cross product of the two edge vectors (np.cross on 2-vectors
        # is deprecated in numpy 2.x, so write it out)
        cross = (b[0] - a[0]) * (c[1] - b[1]) - (b[1] - a[1]) * (c[0] - b[0])
        if np.sign(cross) == orient:          # turns the "wrong" way -> reflex
            return float(b[0]), float(b[1])
    return None


def make_mesh_local(spec, h_glob, h_fine, apex, slope=0.9):
    """
    Mesh with a size field that is fine at `apex` and grows linearly away from it.

        size(x, y) = h_fine + slope * distance_to_apex

    A global refinement to h_fine would be hopelessly expensive; the stress
    singularity is local, so the refinement should be too. `slope` must stay
    gentle enough that the whole FILLET (which sits a radius away from the
    nominal apex) lands inside the fine zone -- too tight a cone refines the
    sharp corner but starves the rounded one, which would fake the result.
    """
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("blk_local")
    try:
        build_solid(pts=spec.get("pts"), ellipse=spec.get("ellipse"),
                    fillet_r=spec["fillet_r"])
        fid = gmsh.model.mesh.field.add("MathEval")
        gmsh.model.mesh.field.setString(
            fid, "F", f"{h_fine} + {slope}*sqrt((x-{apex[0]})^2+(y-{apex[1]})^2)")
        gmsh.model.mesh.field.setAsBackgroundMesh(fid)
        for opt in ["Mesh.MeshSizeFromPoints", "Mesh.MeshSizeFromCurvature",
                    "Mesh.MeshSizeExtendFromBoundary"]:
            gmsh.option.setNumber(opt, 0)     # let the field alone drive sizing
        gmsh.option.setNumber("Mesh.MeshSizeMax", h_glob)
        gmsh.option.setNumber("Mesh.Algorithm3D", 1)
        gmsh.option.setNumber("Mesh.Optimize", 1)
        gmsh.model.mesh.generate(3)
        path = os.path.join(tempfile.gettempdir(), "block_local.msh")
        gmsh.write(path)
    finally:
        gmsh.finalize()
    mio = meshio.read(path)
    tets = np.vstack([c.data for c in mio.cells if c.type == "tetra"])
    return MeshTet(np.ascontiguousarray(mio.points.T),
                   np.ascontiguousarray(tets.T)), tets.shape[0]


APEX = reentrant_vertex(SHAPES["Chevron"]["pts"])
print(f"re-entrant apex of the chevron:       ({APEX[0]:.3f}, {APEX[1]:.3f}) mm")
print(f"fillet radius on the rounded variant:  "
      f"{SHAPES['Rounded chevron']['fillet_r']:.3f} mm")

In [ ]:
# --------------------------------------------------------------------------
# Sample the peak stress ONLY in a small ball around the apex, and only at
# mid-height. The bonded base edge is itself a singularity (bonded-to-free
# transition); if we took a global max it would be contaminated by that edge
# and would say nothing about the corner under test.
# --------------------------------------------------------------------------
APEX_R, Z_LO, Z_HI = 1.5, 0.25 * BLOCK_H, 0.75 * BLOCK_H

def apex_peak(res):
    mesh = res["mesh"]; vm_el = res["A"]["fields"]["vm_el"]
    cen = mesh.p[:, mesh.t].mean(axis=1)
    d = np.hypot(cen[0] - APEX[0], cen[1] - APEX[1])
    sel = (d < APEX_R) & (cen[2] > Z_LO) & (cen[2] < Z_HI)
    return float(vm_el[sel].max())


corner_rows = []
for h_fine in [0.24, 0.12, 0.06, 0.03]:
    for nm in ["Chevron", "Rounded chevron"]:
        m, nt = make_mesh_local(SHAPES[nm], h_glob=2.2, h_fine=h_fine, apex=APEX)
        r = run_shape(nm, SHAPES[nm], verbose=False, mesh=m, n_tet=nt)
        corner_rows.append({
            "h_fine [mm]": h_fine, "variant": nm, "n_tets": nt,
            "vM peak @ apex [MPa]": apex_peak(r),
            "vM_p95 [MPa]": r["A"]["metrics"]["vM_p95 [MPa]"],
        })
corner = pd.DataFrame(corner_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, ttl, logy in zip(
        axes, ["vM peak @ apex [MPa]", "vM_p95 [MPa]"],
        ["PEAK stress at the corner — the singularity test",
         "95th-pct von Mises — the bulk measure"], [True, False]):
    for nm, mk in [("Chevron", "o--"), ("Rounded chevron", "s-")]:
        d = corner[corner["variant"] == nm].sort_values("h_fine [mm]", ascending=False)
        ax.plot(d["h_fine [mm]"], d[col], mk, label=nm)
    ax.set_xscale("log"); ax.invert_xaxis()
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("local element size at the apex [mm]  (refining ->)")
    ax.set_ylabel(col); ax.set_title(ttl); ax.grid(alpha=0.3, which="both"); ax.legend()
fig.suptitle("Sharp vs filleted re-entrant corner — identical footprint area, "
             "identical outline, only the corner radius differs", y=1.03)
fig.tight_layout(); plt.show()

display(corner.pivot(index="h_fine [mm]", columns="variant",
                     values=["n_tets", "vM peak @ apex [MPa]", "vM_p95 [MPa]"])
        .style.format("{:.2f}").set_caption(
            "Local refinement at the re-entrant apex (Load Case A)"))

print(f"Peak von Mises within r < {APEX_R} mm of the apex, "
      f"z in [{Z_LO:.1f}, {Z_HI:.1f}] mm:\n")
for nm in ["Chevron", "Rounded chevron"]:
    d = corner[corner["variant"] == nm].sort_values("h_fine [mm]", ascending=False)
    v = d["vM peak @ apex [MPa]"].to_numpy()
    steps = " -> ".join(f"{x:7.2f}" for x in v)
    pct = "  ".join(f"{100*(v[i+1]-v[i])/v[i]:+5.1f}%" for i in range(len(v) - 1))
    print(f"  {nm:18s} {steps}")
    print(f"  {'':18s} step-to-step: {pct}   overall x{v[-1]/v[0]:.2f}")

**What this shows.** The sharp corner's peak rises steeply and monotonically at every refinement
step, with no sign of settling — it more than doubles across the sweep. The filleted corner rises
at first, while the mesh is still too coarse to see the fillet at all, and then flattens out as the
fillet becomes resolved. Divergent versus bounded: the sharp corner has no peak stress to quote,
and the rounded one does.

That is the durable form of the claim. It survives any change of mesh, whereas "the sharp corner
peaked at X MPa" does not.

Note also that the two variants' **`vM_p95` values stay close together** at every refinement level.
The fillet transforms the *corner* stress and barely touches the *bulk* stress, which is set by the
chevron's slender, spread-out outline rather than by its corner radius. That distinction drives the
interpretation at the end of the notebook — and it is not what the headline peak numbers suggest.

## 7. Visualisation

Three panels per shape, per the specification:

- **A** — deformed shape, coloured by displacement magnitude.
- **B** — von Mises stress on the block surface.
- **C** — contact pressure on the top (asphalt-facing) face.

Surface stress in panel B is evaluated on a `FacetBasis` over the boundary facets, not averaged out
of the interior elements — interior averaging produces a speckled, misleading picture on a coarse
tet mesh.

Colour scales in panels B and C are clipped at the 1st/99th percentile. Without clipping, the single
singular corner element saturates the whole colour range and every plot looks flat and identical.

In [ ]:
def _boundary_vm(sys, x):
    """von Mises on the boundary facets, evaluated there directly (not interpolated out)."""
    mesh = sys["basis"].mesh
    bf = mesh.boundary_facets()
    fb = FacetBasis(mesh, sys["basis"].elem, facets=bf)
    vm = von_mises_of(sigma_of(sym_grad(fb.interpolate(x)))).mean(axis=1)
    return bf, mesh.facets[:, bf].T, vm


def _clip(v, lo=1, hi=99):
    a, b = np.percentile(v, lo), np.percentile(v, hi)
    return (a, b) if b > a else (float(np.min(v)), float(np.min(v)) + 1e-9)


def plot_shape(name, res, case="A"):
    mesh = res["mesh"]; sys = res["sys"]
    x = res[case]["x"]; F = res[case]["fields"]; M = res[case]["metrics"]

    fig = plt.figure(figsize=(15.5, 4.8))
    tri_b = mesh.facets[:, mesh.boundary_facets()].T

    # ---------- Panel A : deformed shape, coloured by |u| --------------------
    L = max(np.ptp(mesh.p[0]), np.ptp(mesh.p[1]), np.ptp(mesh.p[2]))
    # NB: displacements here are LARGER than the block (see the linearity warning),
    # so this factor is a de-magnification, not the usual exaggeration.
    sf = 0.30 * L / max(F["u_mag"].max(), 1e-12)
    pd_ = mesh.p + sf * F["u_nod"]

    ax = fig.add_subplot(131, projection="3d")
    pc = Poly3DCollection(pd_.T[tri_b], cmap="viridis", edgecolor="k", linewidths=0.10)
    pc.set_array(F["u_mag"][tri_b].mean(axis=1))
    ax.add_collection3d(pc)
    for arr, c in zip(pd_, "xyz"):
        getattr(ax, f"set_{c}lim")(arr.min(), arr.max())
    ax.set_box_aspect((np.ptp(pd_[0]), np.ptp(pd_[1]), np.ptp(pd_[2])))
    ax.view_init(elev=22, azim=-58); ax.set_title(
        f"A — deformation  (shown at x{sf:.3f})\n"
        f"$u_{{max}}$ = {M['u_max [mm]']:.2f} mm, mean = {M['u_mean [mm]']:.2f} mm")
    fig.colorbar(pc, ax=ax, shrink=0.62, pad=0.02, label="|u| [mm]")

    # ---------- Panel B : von Mises on the surface ---------------------------
    _, tri_s, vm_s = _boundary_vm(sys, x)
    ax = fig.add_subplot(132, projection="3d")
    pc = Poly3DCollection(mesh.p.T[tri_s], cmap="inferno", edgecolor="k", linewidths=0.10)
    pc.set_array(vm_s); pc.set_clim(*_clip(vm_s))
    ax.add_collection3d(pc)
    for arr, c in zip(mesh.p, "xyz"):
        getattr(ax, f"set_{c}lim")(arr.min(), arr.max())
    ax.set_box_aspect((np.ptp(mesh.p[0]), np.ptp(mesh.p[1]), np.ptp(mesh.p[2])))
    ax.view_init(elev=22, azim=-58); ax.set_title(
        f"B — von Mises (surface)\n"
        f"max = {M['vM_max [MPa]']:.1f}, p95 = {M['vM_p95 [MPa]']:.1f} MPa")
    fig.colorbar(pc, ax=ax, shrink=0.62, pad=0.02, label=r"$\sigma_{vM}$ [MPa]")

    # ---------- Panel C : contact pressure on the top face -------------------
    ax = fig.add_subplot(133)
    tf = mesh.facets[:, sys["top"]].T
    uniq, inv = np.unique(tf.ravel(), return_inverse=True)
    acc = np.zeros(len(uniq)); cnt = np.zeros(len(uniq))
    np.add.at(acc, inv, np.repeat(F["p_fac"], 3)); np.add.at(cnt, inv, 1.0)
    p_nod = acc / np.maximum(cnt, 1.0)
    T = Triangulation(mesh.p[0][uniq], mesh.p[1][uniq], inv.reshape(-1, 3))
    lo, hi = _clip(p_nod)
    cf = ax.tricontourf(T, np.clip(p_nod, lo, hi),
                        levels=np.linspace(lo, hi, 25), cmap="turbo", extend="both")
    ax.triplot(T, color="k", lw=0.12, alpha=0.35)
    ax.set_aspect("equal"); ax.grid(alpha=0.25)
    ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
    ax.set_title(f"C — contact pressure, top face\n"
                 f"max = {M['p_max [MPa]']:.1f}, mean = {M['p_mean [MPa]']:.2f} MPa, "
                 f"CoV = {M['p_CoV [-]']:.2f}")
    fig.colorbar(cf, ax=ax, shrink=0.86, pad=0.02, label=r"$p = -n\cdot\sigma\cdot n$ [MPa]")

    case_lbl = ("A — uniform traction (specified)" if case == "A"
                else "B — rigid platen")
    fig.suptitle(f"{name}   |   Load Case {case_lbl}   |   "
                 f"A_footprint = {res['common']['area_top [mm^2]']:.2f} mm², "
                 f"{res['common']['n_tets']} tets", y=1.02, fontsize=11)
    fig.tight_layout(); plt.show()

## 8. Run all 8 shapes

One parameterised pipeline, called in a loop — no per-shape special cases.

In [ ]:
t_start = time.time()
results, rows_A, rows_B = {}, [], []

print(f"Running {len(SHAPES)} shapes  (h = {MESH_SIZE} mm, P{ELEMENT_ORDER} elements)\n")
for name in SHAPE_ORDER:
    res = run_shape(name, SHAPES[name])
    results[name] = res
    rows_A.append({**res["common"], **res["A"]["metrics"]})
    rows_B.append({**res["common"], **res["B"]["metrics"],
                   "indentation [mm]": res["B"]["info"]["indentation"]})

summary_A = pd.DataFrame(rows_A).set_index("shape")
summary_B = pd.DataFrame(rows_B).set_index("shape")

# carry the geometric descriptors across into both summaries
for df in (summary_A, summary_B):
    df["perimeter [mm]"] = SHAPE_P
    df["shape factor Q [-]"] = SHAPE_Q
    df["tets per mm^3"] = df["n_tets"] / (df["area_top [mm^2]"] * BLOCK_H)

print(f"\nAll shapes solved in {time.time() - t_start:.1f} s")

# ---- fairness check: are the 8 meshes comparably dense? --------------------
dens = summary_A["tets per mm^3"]
spread = dens.max() / dens.min()
print(f"\nMesh density {dens.min():.2f} .. {dens.max():.2f} tets/mm^3 "
      f"(spread {spread:.2f}x)")
if spread > 1.6:
    print("!! WARNING: uneven mesh density across shapes -- peak-stress comparisons\n"
          "   between shapes are NOT trustworthy. Check the mesh sizing options.")
else:
    print("OK - densities are comparable, so cross-shape stress comparison is fair.")

In [ ]:
for name in SHAPE_ORDER:
    plot_shape(name, results[name], case="A")

## 9. Summary — Load Case A (uniform traction, as specified)

Sorted by **max contact pressure**, per the specification.

Read `Fz check` as a solver sanity check: it is the top-face pressure field integrated back over the
face, and it should recover the applied 500 N.

In [ ]:
metric_cols = ["area_top [mm^2]", "shape factor Q [-]", "u_max [mm]", "u_mean [mm]",
               "vM_max [MPa]", "vM_p95 [MPa]",
               "p_max [MPa]", "p_p95 [MPa]", "p_mean [MPa]", "p_CoV [-]",
               "Fz check [N]", "tets per mm^3", "n_tets", "n_dof"]
tblA = summary_A[metric_cols].sort_values("p_max [MPa]", ascending=False)
display(tblA.style.format({c: "{:.3f}" for c in metric_cols[:-2]})
        .background_gradient(cmap="Reds", subset=["vM_p95 [MPa]", "p_max [MPa]", "p_CoV [-]"])
        .set_caption("Load Case A — all metrics, sorted by max contact pressure"))

sp = summary_A["area_top [mm^2]"]
print(f"Footprint area across all 8 shapes: {sp.min():.4f} .. {sp.max():.4f} mm^2 "
      f"(spread {100*(sp.max()-sp.min())/AREA_T:.2f} %) -> shape is the only variable.")
print("  (The small deficit on the curved shapes is the MESHED area: flat triangles")
print("   chord-approximate a curved outline from the inside. The OCC areas above are")
print("   exact and identical; this is a discretisation effect, not a geometry error.)")
print(f"\nFz recovery: {summary_A['Fz check [N]'].min():.1f} .. "
      f"{summary_A['Fz check [N]'].max():.1f} N vs {LOAD_N:.0f} N applied "
      f"(max deviation {100*max(abs(summary_A['Fz check [N]']-LOAD_N))/LOAD_N:.1f} %) "
      "-- the recovered traction integrates back to the applied load, as it must.")

### 9b. Does compactness explain the ranking?

The shape factor $Q = P^2/(4\pi A)$ measures how much perimeter a shape carries for a fixed area
($Q=1$ for a circle). With area pinned, it is the natural single-number description of "how spread
out" a cross-section is — and it is worth asking whether it, rather than corner sharpness, is what
actually drives the stress ranking.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
for ax, col in zip(axes, ["vM_p95 [MPa]", "u_max [mm]", "p_CoV [-]"]):
    q, y = summary_A["shape factor Q [-]"], summary_A[col]
    ax.scatter(q, y, s=64, c=["tab:red" if n in ("Chevron", "Rounded chevron")
                              else "tab:blue" for n in summary_A.index],
               edgecolor="k", zorder=3)
    for n in summary_A.index:
        ax.annotate(n, (q[n], y[n]), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")
    rho = q.rank().corr(y.rank(), method="spearman")
    ax.set_xlabel("shape factor  Q = $P^2/4\\pi A$"); ax.set_ylabel(col)
    ax.set_title(f"{col}\nSpearman $\\rho$ = {rho:+.3f}"); ax.grid(alpha=0.3)
fig.suptitle("Compactness vs response — area is identical, so Q is the shape variable", y=1.03)
fig.tight_layout(); plt.show()

print("Spearman rank correlation with shape factor Q:")
for col in ["vM_p95 [MPa]", "vM_max [MPa]", "u_max [mm]", "p_CoV [-]", "p_max [MPa]"]:
    rho = summary_A["shape factor Q [-]"].rank().corr(
        summary_A[col].rank(), method="spearman")
    print(f"   Q vs {col:16s} rho = {rho:+.3f}")

In [ ]:
# ---- grouped bar chart: the three headline comparisons ----------------------
plot_cols = [("vM_p95 [MPa]", "95th-pct von Mises [MPa]", "tab:red"),
             ("u_max [mm]",   "max deformation [mm]",     "tab:blue"),
             ("p_CoV [-]",    "contact-pressure CoV [-]", "tab:green")]

order = summary_A["vM_p95 [MPa]"].sort_values(ascending=False).index
xpos = np.arange(len(order)); width = 0.26

fig, ax = plt.subplots(figsize=(13, 4.6))
axes2 = [ax, ax.twinx(), ax.twinx()]
axes2[2].spines["right"].set_position(("outward", 52))
for k, ((col, lbl, colr), a) in enumerate(zip(plot_cols, axes2)):
    a.bar(xpos + (k - 1) * width, summary_A.loc[order, col], width,
          color=colr, alpha=0.85, label=lbl, edgecolor="k", linewidth=0.4)
    a.set_ylabel(lbl, color=colr); a.tick_params(axis="y", colors=colr)
ax.set_xticks(xpos); ax.set_xticklabels(order, rotation=18, ha="right")
ax.set_title("Load Case A — shape comparison at identical footprint area "
             "(three independent axes; compare within a colour, not across)")
ax.grid(axis="y", alpha=0.25)
h = [a.patches[0] for a in axes2]
ax.legend(h, [c[1] for c in plot_cols], loc="upper right", framealpha=0.95)
fig.tight_layout(); plt.show()

In [ ]:
# ---- normalised view: everything relative to the Square baseline ------------
base = "Square"
norm_cols = ["u_max [mm]", "vM_p95 [MPa]", "p_max [MPa]", "p_CoV [-]"]
rel = summary_A[norm_cols].div(summary_A.loc[base, norm_cols])
rel = rel.loc[summary_A["vM_p95 [MPa]"].sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 4.4))
xpos = np.arange(len(rel)); width = 0.2
for k, c in enumerate(norm_cols):
    ax.bar(xpos + (k - 1.5) * width, rel[c], width, label=c,
           edgecolor="k", linewidth=0.4)
ax.axhline(1.0, color="k", lw=1.2, ls="--")
ax.text(len(rel) - 0.4, 1.03, f"{base} = 1.0", ha="right", fontsize=8)
ax.set_xticks(xpos); ax.set_xticklabels(rel.index, rotation=18, ha="right")
ax.set_ylabel(f"ratio to {base}")
ax.set_title(f"Load Case A — all metrics normalised to the {base} baseline "
             "(dimensionless, so load-independent)")
ax.grid(axis="y", alpha=0.25); ax.legend(ncol=4, fontsize=8)
fig.tight_layout(); plt.show()

display(rel.style.format("{:.3f}").background_gradient(cmap="RdYlGn_r")
        .set_caption(f"Load Case A — metrics relative to {base}"))

## 10. Load Case B — contact pressure as an *output*

Load Case A imposes a uniform pressure, so its pressure map mostly reflects corner singularities
rather than a genuine footprint pressure distribution. Load Case B presses each block with a
**rigid flat platen**, which lets the pressure distribute itself — the physically meaningful
comparison for footprint uniformity.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7.6))
for ax, name in zip(axes.ravel(), SHAPE_ORDER):
    res = results[name]; mesh = res["mesh"]
    F = res["B"]["fields"]; M = res["B"]["metrics"]
    tf = mesh.facets[:, res["sys"]["top"]].T
    uniq, inv = np.unique(tf.ravel(), return_inverse=True)
    acc = np.zeros(len(uniq)); cnt = np.zeros(len(uniq))
    np.add.at(acc, inv, np.repeat(F["p_fac"], 3)); np.add.at(cnt, inv, 1.0)
    p_nod = acc / np.maximum(cnt, 1.0)
    T = Triangulation(mesh.p[0][uniq], mesh.p[1][uniq], inv.reshape(-1, 3))
    lo, hi = _clip(p_nod)
    cf = ax.tricontourf(T, np.clip(p_nod, lo, hi),
                        levels=np.linspace(lo, hi, 22), cmap="turbo", extend="both")
    ax.set_aspect("equal"); ax.set_xlim(-11, 11); ax.set_ylim(-11, 13)
    ax.set_title(f"{name}\nCoV = {M['p_CoV [-]']:.2f}, "
                 f"$p_{{max}}$ = {M['p_max [MPa]']:.1f} MPa", fontsize=9)
    fig.colorbar(cf, ax=ax, shrink=0.80, pad=0.02)
fig.suptitle("Load Case B — rigid-platen contact pressure. Sliding drag acts in +x; "
             "note the edge concentration (flat-punch effect).", y=1.01)
fig.tight_layout(); plt.show()

In [ ]:
colsB = ["u_max [mm]", "vM_p95 [MPa]", "p_max [MPa]", "p_p95 [MPa]",
         "p_mean [MPa]", "p_min [MPa]", "p_CoV [-]", "indentation [mm]"]
tblB = summary_B[colsB].sort_values("p_max [MPa]", ascending=False)
display(tblB.style.format("{:.3f}")
        .background_gradient(cmap="Reds", subset=["p_max [MPa]", "p_CoV [-]"])
        .set_caption("Load Case B — rigid platen, sorted by max contact pressure"))

neg = summary_B[summary_B["p_min [MPa]"] < 0]
if len(neg):
    print("Shapes showing NEGATIVE pressure (tension) under the bonded platen.")
    print("A real block would separate there; the bonded BC holds it down instead:")
    for n, v in neg["p_min [MPa]"].items():
        print(f"   {n:18s} p_min = {v:8.3f} MPa")

In [ ]:
# ---- do the two load cases agree on the ranking? ---------------------------
rank = pd.DataFrame({
    "rank A (vM_p95)": summary_A["vM_p95 [MPa]"].rank(ascending=False).astype(int),
    "rank B (vM_p95)": summary_B["vM_p95 [MPa]"].rank(ascending=False).astype(int),
    "rank A (p_CoV)":  summary_A["p_CoV [-]"].rank(ascending=False).astype(int),
    "rank B (p_CoV)":  summary_B["p_CoV [-]"].rank(ascending=False).astype(int),
}).sort_values("rank A (vM_p95)")
display(rank.style.set_caption("Shape ranking under both load cases (1 = worst / highest)"))

for a, b, lbl in [("vM_p95 [MPa]", "vM_p95 [MPa]", "vM_p95"),
                  ("p_CoV [-]", "p_CoV [-]", "p_CoV")]:
    rho = summary_A[a].rank().corr(summary_B[b].rank(), method="spearman")
    print(f"Spearman rank correlation between Load Case A and B for {lbl}: {rho:+.3f}")

## 11. Interpretation

*The cell below reads the actual computed numbers rather than restating an expectation, so the
conclusions stay correct if you change the material, the load, or the mesh.*

In [ ]:
sa = summary_A
rank_vm  = sa["vM_p95 [MPa]"].sort_values(ascending=False)
rank_cov = sa["p_CoV [-]"].sort_values(ascending=False)
rank_u   = sa["u_max [mm]"].sort_values(ascending=False)

ROUNDED = ["Ellipse", "Rounded rect", "Rounded chevron"]
SHARP   = [s for s in SHAPE_ORDER if s not in ROUNDED]
CHEVRONS = ["Chevron", "Rounded chevron"]
COMPACT  = [s for s in SHAPE_ORDER if s not in CHEVRONS]

print("=" * 78)
print("STRESS BUILD-UP  (95th-pct von Mises, volume-weighted)")
print("=" * 78)
for i, (n, v) in enumerate(rank_vm.items(), 1):
    tag = f"{'rounded' if n in ROUNDED else 'sharp  '} / Q={sa.loc[n,'shape factor Q [-]']:.2f}"
    print(f"  {i}. {n:18s} {v:8.3f} MPa   [{tag}]")
print(f"\n  spread worst/best = {rank_vm.iloc[0]/rank_vm.iloc[-1]:.2f}x")

print("\n  Grouped two ways -- the grouping that matters is the second one:")
print(f"    by CORNER:      sharp   {sa.loc[SHARP,    'vM_p95 [MPa]'].mean():7.3f}   "
      f"rounded {sa.loc[ROUNDED,  'vM_p95 [MPa]'].mean():7.3f} MPa   "
      f"(ratio {sa.loc[SHARP,'vM_p95 [MPa]'].mean()/sa.loc[ROUNDED,'vM_p95 [MPa]'].mean():.2f}x)")
print(f"    by COMPACTNESS: compact {sa.loc[COMPACT,  'vM_p95 [MPa]'].mean():7.3f}   "
      f"chevron {sa.loc[CHEVRONS, 'vM_p95 [MPa]'].mean():7.3f} MPa   "
      f"(ratio {sa.loc[CHEVRONS,'vM_p95 [MPa]'].mean()/sa.loc[COMPACT,'vM_p95 [MPa]'].mean():.2f}x)")

print("\n" + "=" * 78)
print("CONTACT-PRESSURE NON-UNIFORMITY  (CoV, higher = less uniform)")
print("=" * 78)
for i, (n, v) in enumerate(rank_cov.items(), 1):
    print(f"  {i}. {n:18s} CoV = {v:7.3f}   p_max = {sa.loc[n,'p_max [MPa]']:8.3f} MPa")

print("\n" + "=" * 78)
print("DEFORMATION  (max displacement magnitude)")
print("=" * 78)
for i, (n, v) in enumerate(rank_u.items(), 1):
    print(f"  {i}. {n:18s} {v:8.3f} mm   ({v/BLOCK_H:5.2f} x block height)")

print("\n" + "=" * 78)
print("CHEVRON: effect of filleting the re-entrant corner  (equal area)")
print("=" * 78)
print("Identical outline, identical area -- the ONLY difference is the corner")
print("radius. This is the cleanest controlled comparison in the study.\n")
RESOLVED = {"vM_p95 [MPa]", "p_CoV [-]", "u_max [mm]", "p_mean [MPa]"}
for c in ["vM_p95 [MPa]", "u_max [mm]", "p_CoV [-]", "vM_max [MPa]", "p_max [MPa]"]:
    s_, r_ = sa.loc["Chevron", c], sa.loc["Rounded chevron", c]
    flag = "" if c in RESOLVED else "   <-- NOT MESH-RESOLVED, see below"
    print(f"  {c:16s} sharp {s_:9.3f} -> rounded {r_:9.3f}   "
          f"({100*(r_-s_)/s_:+7.1f} %){flag}")

print(f"""
  Reading this table:

  * The first three rows are volume/area-weighted and mesh-converged at this
    resolution, so their signs and magnitudes are trustworthy. They say the
    fillet leaves bulk stress essentially unchanged ({100*(sa.loc['Rounded chevron','vM_p95 [MPa]']-sa.loc['Chevron','vM_p95 [MPa]'])/sa.loc['Chevron','vM_p95 [MPa]']:+.1f} %) while
    modestly reducing compliance and pressure scatter.

  * The last two rows are PEAK values on the production mesh (h = {MESH_SIZE} mm),
    which is far too coarse to resolve either the sharp apex or the {SHAPES['Rounded chevron']['fillet_r']:.2f} mm
    fillet. At this resolution they mostly report which variant happened to
    get more elements near the corner -- which is why they come out with the
    'wrong' sign, suggesting that rounding RAISES peak stress.

    That is an artefact, and section 6b is the answer to it: with local
    refinement at the apex, the sharp corner's peak diverges without limit
    while the filleted corner converges to a finite value. Trust that study
    for the peak-stress question, not these two rows.""")

### What the numbers say

The headline result is **not** the one this study was set up to expect. The original hypothesis was
that sharp-cornered shapes (chevron, square, pentagon) would show higher stress than rounded ones at
equal area. The data only partly supports that, and the part it contradicts is the more interesting
half.

**1. Compactness dominates, not corner sharpness.**
Sorting by bulk stress (`vM_p95`) does not separate sharp shapes from rounded shapes — it separates
**chevrons from everything else**. Both chevron variants sit far above the field; the square,
hexagon, pentagon and trapezoid — all sharp-cornered — sit in a tight band barely above the ellipse
and the rounded rectangle. Group the shapes by corner treatment and the two means are nearly
identical; group them by compactness and a large gap opens — roughly a **1.6× jump in bulk stress**,
against a corner-treatment effect of order 1%. The shape factor $Q=P^2/4\pi A$ correlates positively
with every response metric, and it separates the two chevrons from the six compact shapes cleanly.

(The rank correlation is moderate rather than near-perfect, and that is expected: the six compact
shapes sit inside a narrow band of both $Q$ and stress, so their relative order is essentially noise.
The signal is the group separation, not the fine ordering within the compact cluster.)

The mechanism is straightforward. The chevron is slender and spread out; its arms act as cantilevers
under the friction drag and accumulate bending stress along their length. That is a *global*
stiffness effect distributed over the whole body, and it dwarfs whatever happens in the small region
around a corner.

**2. Corner radius governs the singular peak — a real effect, but a local one.**
The sharp-vs-filleted convergence study makes the honest version of the claim: at the sharp
re-entrant apex the peak stress **diverges** under mesh refinement, while the filleted corner
**converges to a finite value**. That is a genuine and important difference — an unbounded peak is
where a crack initiates — but it lives in a small volume, which is exactly why the two variants'
volume-weighted `vM_p95` values sit almost on top of each other.

So both statements are true and they are not in conflict:
*fillets fix the corner; they do not fix the shape.*

**3. This is why the metric had to be chosen before the comparison.**
`vM_max` and `p_max` grow without limit at any sharp corner, so a table of maxima ranks **meshes**,
not shapes. This is not hypothetical: enabling curvature-adaptive meshing refines the filleted
shapes 3–4× harder and produces the confident, wrong conclusion that *rounding a corner raises peak
stress*. Uniform mesh density plus volume-weighted percentiles is what makes the ranking mean
anything.

**4. Contact pressure is only weakly informative in Load Case A — by construction.**
Imposing a uniform normal traction largely determines the recovered normal traction, so Load Case A's
`p_CoV` mostly reflects corner singularities rather than footprint mechanics. **Read Load Case B for
pressure uniformity**: there the flat-punch edge concentration develops freely, and the shapes with
the most perimeter per unit area show the least uniform footprint — the same compactness story.

**5. Every block deforms enormously, and the chevron most of all.**
Displacements come out at several times the block height, which is the linearity warning made
visible rather than a physical prediction. The *ratios* are still meaningful, and they track $Q$:
the chevron's slender arms make it by far the most compliant shape at equal footprint area.

### Design read-across (with the caveats intact)

- At equal footprint area, **compactness buys stress margin** far more effectively than corner
  radius. A designer choosing between a chevron and a compact lug is making a much bigger mechanical
  decision than one choosing a fillet radius.
- **Fillet the re-entrant corners anyway.** The bulk stress barely moves, but an unbounded local peak
  becomes bounded, and that is where fatigue cracks start. It is cheap insurance, not a bulk fix.
- Real tread design does not optimise for low stress alone — chevrons and sharp lugs exist because
  biting edges deliver wet and snow traction, and void geometry clears water. This study quantifies
  the **mechanical cost** of those features; it does not argue against them.
- Absolute magnitudes here are **not** design numbers. The load produces ~100% nominal strain in a
  linear model (see the sanity check), and rubber at that strain is strongly nonlinear. Rankings are
  load-independent by linearity; magnitudes are not.

### If you want to take this further

1. Drop `LOAD_N` to ~50 N for magnitudes inside the linear regime — rankings will not move.
2. Swap the linear material for Neo-Hookean or Mooney–Rivlin with a real contact solve
   (`FEniCSx`, `CalculiX`, or a commercial code) once relative screening has narrowed the candidates.
3. Sweep the fillet radius on the chevron to find the knee in the stress-vs-radius curve.
4. Add the neighbouring blocks and the groove geometry — real blocks lean on each other, which
   changes the pressure distribution materially.

In [ ]:
# ---- persist the results so they can be reused without re-solving ----------
summary_A.to_csv("summary_load_case_A.csv")
summary_B.to_csv("summary_load_case_B.csv")
area_df.to_csv("area_matching.csv")
print("Wrote: summary_load_case_A.csv, summary_load_case_B.csv, area_matching.csv")
print(f"\nTotal wall time for the study: {time.time() - t_start:.1f} s")